# 06 - Save the Final Model

Goal of this notebook: train the FINAL version of our model (TF-IDF +
calibrated SVM, as decided in notebook 05) on ALL the data we're allowed to
use for training, and save it to disk so the API service (`src/app.py`) can
load it later.

## Why retrain here instead of just reusing notebook 05's model?

In notebook 04/05 we only trained on the **train** split (80%), keeping
val/test separate to fairly evaluate the model. Now that we've already
picked our final approach and evaluated it honestly, it's good practice to
do one **final training run on train + validation combined** (90% of data)
before saving - this squeezes a bit more learning out of the data we're
allowed to use for training, while the test set stays untouched as our
already-reported honest score from notebook 05.


## Step 1: Load train and validation sets, combine them


In [1]:
import pandas as pd

train_df = pd.read_csv("../data/processed/train.csv").fillna({"text": ""})
val_df = pd.read_csv("../data/processed/val.csv").fillna({"text": ""})

# Combine train + val for the final model (test stays separate and untouched)
final_train_df = pd.concat([train_df, val_df], ignore_index=True)

print("Final training set size:", len(final_train_df))

Final training set size: 20939


## Step 2: Fit the TF-IDF vectorizer on this combined data

Same settings as before (`max_features=20000`, `ngram_range=(1,2)`, `min_df=2`).


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

X_final = vectorizer.fit_transform(final_train_df["text"])
y_final = final_train_df["label"]

print("Final TF-IDF matrix shape:", X_final.shape)

Final TF-IDF matrix shape: (20939, 5050)


## Step 3: Train the final calibrated SVM

This is the exact model type we decided on in notebook 05 - Linear SVM
wrapped in `CalibratedClassifierCV` so it gives trustworthy confidence scores.


In [3]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

final_model = CalibratedClassifierCV(
    LinearSVC(class_weight="balanced", max_iter=5000),
    cv=5
)
final_model.fit(X_final, y_final)

print("Final model training complete")

Final model training complete


## Step 4: Quick sanity check before saving

Let's test it on a couple of sample sentences to make sure it behaves as
expected before we save it.


In [4]:
samples = [
    "i want to track my refund",
    "how do i cancel my subscription",
    "my package hasn't arrived yet",
]

for s in samples:
    vec = vectorizer.transform([s])
    probs = final_model.predict_proba(vec)[0]
    idx = probs.argmax()
    print(f"'{s}' -> {final_model.classes_[idx]}  (confidence: {probs[idx]:.4f})")

'i want to track my refund' -> REFUND  (confidence: 0.9946)
'how do i cancel my subscription' -> SUBSCRIPTION  (confidence: 0.8647)
'my package hasn't arrived yet' -> ACCOUNT  (confidence: 0.7563)


## Step 5: Save the model and vectorizer to disk

We use `joblib` to save Python objects to files. **Both files are needed
together** at prediction time - the vectorizer to convert new text into
numbers, and the model to make a prediction from those numbers.


In [5]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(final_model, "../models/model.pkl")
joblib.dump(vectorizer, "../models/vectorizer.pkl")

print("Saved model.pkl and vectorizer.pkl to models/")

Saved model.pkl and vectorizer.pkl to models/


## Step 6: Verify the saved files load correctly

Always good practice to reload what you just saved and confirm it still works -
catches any saving mistakes immediately instead of discovering them later.


In [6]:
loaded_model = joblib.load("../models/model.pkl")
loaded_vectorizer = joblib.load("../models/vectorizer.pkl")

test_sentence = "i need help with my invoice payment"
vec = loaded_vectorizer.transform([test_sentence])
probs = loaded_model.predict_proba(vec)[0]
idx = probs.argmax()

print(f"Reloaded model prediction for '{test_sentence}':")
print(f"-> {loaded_model.classes_[idx]}  (confidence: {probs[idx]:.4f})")

Reloaded model prediction for 'i need help with my invoice payment':
-> INVOICE  (confidence: 0.5349)


## Summary / What we learned

- Trained the FINAL model on train+validation combined (test set was never
  touched here - its honest score was already reported in notebook 05)
- Used the calibrated SVM so confidence scores are trustworthy
- Saved `model.pkl` and `vectorizer.pkl` to the `models/` folder
- Verified the saved files reload and work correctly

**This is the same `models/model.pkl` and `models/vectorizer.pkl` that
`src/predict.py` and `src/app.py` already expect** - so once this notebook
runs successfully, your FastAPI service in `src/app.py` is ready to serve
real predictions.

**Next step:** go back to `src/app.py` and start the API:
```
cd ../src
uvicorn app:app --reload --port 8000
```
Then test it with the curl command from before. This is the final step of
Phase 1 (ml-service) - after this, we move to Spring Boot.
